# Refrigerated Vehicles with AMPL in Python
## AMPL Modeling for a Small Transportation MILP

This notebook solves the refrigerated-vehicles problem with AMPL from Python using `amplpy`.

We will solve:
- the base problem from book section `2.4`

Each run reports:
- the best solution found
- the cost
- the execution time


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [9]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [10]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def extract_solution(variable, variable_order):
    values = variable.get_values().to_dict()
    solution = {}

    for name in variable_order:
        if name in values:
            solution[name] = int(round(values[name]))
        else:
            solution[name] = int(round(values[(name,)]))

    return solution


def extract_matrix_solution(variable):
    values = variable.get_values().to_dict()
    solution = {}

    for key, value in values.items():
        if abs(value) > 1e-9:
            solution[key] = int(round(value))

    return solution


## Problem: Refrigerated Vehicles

Variables:
- `type_a_trucks`
- `type_b_trucks`

Objective:
- minimize rental cost


In [12]:
VEHICLE_EXPECTED = {
    "type_a_trucks": 15,
    "type_b_trucks": 20,
    "cost": 1250,
}


@timed
def solve_refrigerated_vehicles_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var type_a_trucks >= 0 integer;
        var type_b_trucks >= 0 integer;

        minimize Cost:
            30 * type_a_trucks + 40 * type_b_trucks;

        subject to RefrigeratedCapacity:
            20 * type_a_trucks + 30 * type_b_trucks >= 900;

        subject to NonRefrigeratedCapacity:
            40 * type_a_trucks + 30 * type_b_trucks >= 1200;
        '''
    )
    ampl.solve(solver=solver)

    solution = {
        "type_a_trucks": int(round(ampl.get_value("type_a_trucks"))),
        "type_b_trucks": int(round(ampl.get_value("type_b_trucks"))),
        "cost": int(round(ampl.get_value("Cost"))),
    }
    return solution


In [13]:
vehicle_result, vehicle_time = solve_refrigerated_vehicles_with_ampl()

print("=== REFRIGERATED VEHICLES RESULTS WITH AMPL ===")
print(f"Solution -> {vehicle_result}")
print(f"Time     -> {vehicle_time:.8f} seconds")

assert vehicle_result == VEHICLE_EXPECTED
print("The AMPL solution matches the textbook optimum.")


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === REFRIGERATED VEHICLES RESULTS WITH AMPL ===
Solution -> {'type_a_trucks': 15, 'type_b_trucks': 20, 'cost': 1250}
Time     -> 1.60982812 seconds
The AMPL solution matches the textbook optimum.


## Variant Note

The source text does not include a separate student variant for section `2.4`, so this notebook focuses on the published base formulation only.
